# Notebook 09b — Datacube NetCDF extendido (CF + EPSG:9377)

**Versión extendida** del notebook 09. Genera hasta tres cubos NetCDF según los TIFs disponibles localmente:

1. `cgsm_datacube_periodos.nc` — 3 RGB de referencia (degradación, recuperación, actual)
2. `cgsm_datacube_trimestral.nc` — composites trimestrales Sentinel-2 si están en `data/processed/s2/`
3. `cgsm_datacube_landsat.nc` — composites anuales Landsat si están en `data/processed/landsat/`

Todos los cubos quedan en EPSG:9377 (MAGNA-SIRGAS Origen Nacional) con coordenadas en metros.

In [1]:
import re
from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray
import pandas as pd
from rasterio.enums import Resampling

ROOT = Path('..').resolve()
RGB_DIR = ROOT / 'data' / 'processed' / 'rgb_acotado'
S2_DIR  = ROOT / 'data' / 'processed' / 's2'
LS_DIR  = ROOT / 'data' / 'processed' / 'landsat'
OUT_DIR = ROOT / 'data' / 'processed' / 'cubo'
OUT_DIR.mkdir(parents=True, exist_ok=True)

ATTRS_BASE = {
    'Conventions': 'CF-1.8',
    'institution': 'Universidad Nacional de Colombia, Maestria en Geomatica',
    'creator_name': 'Lina Maria Quintero Fonseca',
    'creator_email': 'linaq112008@gmail.com',
    'project': 'Proyecto final Programacion en SIG, UNAL 2026-1',
    'crs_origen': 'EPSG:9377 MAGNA-SIRGAS Origen Nacional',
    'references': 'Hersbach et al. 2020; Bunting et al. 2022; Wu y Osco 2023',
    'geospatial_lat_min':  10.30, 'geospatial_lat_max':  11.20,
    'geospatial_lon_min': -75.05, 'geospatial_lon_max': -74.05,
}

print(f'rgb_acotado/: {len(list(RGB_DIR.glob("*.tif"))) if RGB_DIR.exists() else 0} TIFs')
print(f's2/:          {len(list(S2_DIR.glob("*.tif")))  if S2_DIR.exists()  else 0} TIFs')
print(f'landsat/:     {len(list(LS_DIR.glob("*.tif")))  if LS_DIR.exists()  else 0} TIFs')

rgb_acotado/: 3 TIFs
s2/:          31 TIFs
landsat/:     13 TIFs


In [3]:
def cargar_y_reproyectar(path, fecha):
    da = rioxarray.open_rasterio(path, chunks={'x': 1024, 'y': 1024})
    if str(da.rio.crs).upper() != 'EPSG:9377':
        da = da.rio.reproject('EPSG:9377', resampling=Resampling.bilinear)
    return da.assign_coords(time=fecha).expand_dims('time')

def alinear(rasters_dict):
    ref = list(rasters_dict.values())[0].isel(time=0)
    out = {}
    for k, da in rasters_dict.items():
        d = da.isel(time=0)
        if (d.sizes['x'], d.sizes['y']) != (ref.sizes['x'], ref.sizes['y']):
            d = d.rio.reproject_match(ref, resampling=Resampling.bilinear)
        out[k] = d.assign_coords(time=da['time'].values[0]).expand_dims('time')
    return out

def guardar(ds, path, extra=None):
    ds = ds.rio.write_crs('EPSG:9377')
    ds.attrs.update(ATTRS_BASE)
    if extra: ds.attrs.update(extra)
    ds.attrs['history'] = pd.Timestamp.now().strftime('%Y-%m-%d') + ' build by 09b'
    if path.exists(): path.unlink()
    enc = {v: {'zlib': True, 'complevel': 4} for v in ds.data_vars}
    ds.to_netcdf(path, encoding=enc, format='NETCDF4')
    return path.stat().st_size / (1024*1024)

## 1. Cubo de períodos (3 RGB acotados)

In [4]:
PERIODOS = {
    'degradacion':  pd.Timestamp('2020-09-15'),
    'recuperacion': pd.Timestamp('2022-03-15'),
    'actual':       pd.Timestamp('2024-12-15'),
}
rasters = {}
for nombre, fecha in PERIODOS.items():
    p = RGB_DIR / f'CGSM_RGB_{nombre}.tif'
    if not p.exists():
        print(f'  no encontrado: {p}'); continue
    rasters[nombre] = cargar_y_reproyectar(p, fecha)
    print(f'  {nombre:13s} reproyectado, shape={tuple(rasters[nombre].sizes.values())}')

if rasters:
    rasters = alinear(rasters)
    cubo = xr.concat(list(rasters.values()), dim='time')
    cubo = cubo.rename({'band': 'band_idx'})
    cubo = cubo.assign_coords(band_idx=['B4_red','B3_green','B2_blue'])
    cubo.name = 'reflectance_byte'
    cubo['x'].attrs.update({'long_name':'Coordenada X EPSG:9377','units':'meters'})
    cubo['y'].attrs.update({'long_name':'Coordenada Y EPSG:9377','units':'meters'})
    out = OUT_DIR / 'cgsm_datacube_periodos.nc'
    tam = guardar(cubo.to_dataset(), out, {
        'title':'Datacube periodos de referencia CGSM 2020-2025',
        'comment':'Tres composites RGB sobre AOI acotado SFF+VPI (835 km2), insumo SamGeo.',
    })
    print(f'\nGuardado: {out.name}  ({tam:.1f} MB)')

  degradacion   reproyectado, shape=(1, 3, 5879, 5922)
  recuperacion  reproyectado, shape=(1, 3, 5879, 5922)
  actual        reproyectado, shape=(1, 3, 5879, 5922)

Guardado: cgsm_datacube_periodos.nc  (39.7 MB)


## 2. Cubo trimestral Sentinel-2 (auto-detección)

In [ ]:
tifs_s2 = sorted(S2_DIR.glob('CGSM_indices_*.tif')) if S2_DIR.exists() else []
if not tifs_s2:
    print(f'Sin TIFs trimestrales en {S2_DIR}. Cubo omitido.')
else:
    pat = re.compile(r'(\d{4})[_-]?Q(\d)')
    rasters_s2 = {}
    for p in tifs_s2:
        m = pat.search(p.name)
        if not m: continue
        año, q = int(m.group(1)), int(m.group(2))
        mes = (q - 1) * 3 + 2
        fecha = pd.Timestamp(f'{año}-{mes:02d}-15')
        rasters_s2[f'{año}_Q{q}'] = cargar_y_reproyectar(p, fecha)
        print(f'  {año}-Q{q} reproyectado → {fecha.date()}')
    
    rasters_s2 = alinear(rasters_s2)
    cubo_s2 = xr.concat(list(rasters_s2.values()), dim='time')
    cubo_s2 = cubo_s2.rename({'band':'band_idx'})
    cubo_s2 = cubo_s2.assign_coords(band_idx=['NDVI','NDWI','CMRI'])
    cubo_s2.name = 'indices_espectrales'
    cubo_s2['x'].attrs.update({'long_name':'Coordenada X EPSG:9377','units':'meters'})
    cubo_s2['y'].attrs.update({'long_name':'Coordenada Y EPSG:9377','units':'meters'})
    out = OUT_DIR / 'cgsm_datacube_trimestral.nc'
    tam = guardar(cubo_s2.to_dataset(), out, {
        'title': f'Datacube trimestral CGSM 2018-2025 ({len(rasters_s2)} trimestres)',
        'comment': 'Composites trimestrales NDVI/NDWI/CMRI Sentinel-2 sobre AOI acotado.',
    })
    print(f'\nGuardado: {out.name}  ({tam:.1f} MB)')
    print(f'Dimensiones: {dict(cubo_s2.sizes)}')

  2018-Q1 reproyectado → 2018-02-15
  2018-Q4 reproyectado → 2018-11-15
  2019-Q1 reproyectado → 2019-02-15
  2019-Q2 reproyectado → 2019-05-15
  2019-Q3 reproyectado → 2019-08-15
  2019-Q4 reproyectado → 2019-11-15
  2020-Q1 reproyectado → 2020-02-15
  2020-Q2 reproyectado → 2020-05-15
  2020-Q3 reproyectado → 2020-08-15
  2020-Q4 reproyectado → 2020-11-15
  2021-Q1 reproyectado → 2021-02-15


## 3. Cubo anual Landsat 8/9 (auto-detección)

In [ ]:
tifs_ls = sorted(LS_DIR.glob('CGSM_Landsat_indices_*.tif')) if LS_DIR.exists() else []
if not tifs_ls:
    print(f'Sin TIFs Landsat en {LS_DIR}. Cubo omitido.')
else:
    pat = re.compile(r'(\d{4})')
    rasters_ls = {}
    for p in tifs_ls:
        m = pat.search(p.name)
        if not m: continue
        año = int(m.group(1))
        fecha = pd.Timestamp(f'{año}-07-01')
        rasters_ls[str(año)] = cargar_y_reproyectar(p, fecha)
        print(f'  {año} reproyectado → {fecha.date()}')
    
    rasters_ls = alinear(rasters_ls)
    cubo_ls = xr.concat(list(rasters_ls.values()), dim='time')
    cubo_ls = cubo_ls.rename({'band':'band_idx'})
    cubo_ls = cubo_ls.assign_coords(band_idx=['NDVI','NDWI','CMRI'])
    cubo_ls.name = 'indices_espectrales'
    cubo_ls['x'].attrs.update({'long_name':'Coordenada X EPSG:9377','units':'meters'})
    cubo_ls['y'].attrs.update({'long_name':'Coordenada Y EPSG:9377','units':'meters'})
    out = OUT_DIR / 'cgsm_datacube_landsat.nc'
    tam = guardar(cubo_ls.to_dataset(), out, {
        'title': f'Datacube anual Landsat CGSM 2013-2025 ({len(rasters_ls)} anos)',
        'comment': 'Composites anuales NDVI/NDWI/CMRI Landsat 8/9 sobre AOI acotado.',
    })
    print(f'\nGuardado: {out.name}  ({tam:.1f} MB)')
    print(f'Dimensiones: {dict(cubo_ls.sizes)}')

## 4. Verificación final

In [ ]:
for nc in sorted(OUT_DIR.glob('*.nc')):
    ds = xr.open_dataset(nc, chunks={'time': 1}, decode_coords='all')
    print(f'\n=== {nc.name} ===')
    print(f'  Dimensiones: {dict(ds.sizes)}')
    print(f'  Variables:   {list(ds.data_vars)}')
    try:
        print(f'  CRS:         {ds.rio.crs}')
        print(f'  X range:     {float(ds.x.min()):.0f} a {float(ds.x.max()):.0f} m')
        print(f'  Y range:     {float(ds.y.min()):.0f} a {float(ds.y.max()):.0f} m')
    except Exception as e:
        print(f'  (CRS info no disponible: {e})')
    print(f'  Tamano:      {nc.stat().st_size / (1024*1024):.1f} MB')
    ds.close()